In [79]:
# Some practice with parsing Fortran code, building ASTs, editing code, and more with Processor class

In [80]:
%load_ext autoreload
%autoreload 2
from fparser.two import Fortran2003 as F23
from fparser.two import Fortran2008 as F28
from fparser.two.utils import walk
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [81]:
# change to the Fgpt directory
%cd /home/kardaneh/Fgpt

/home/kardaneh/Fgpt


/home/kardaneh/fparser_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [82]:
# Processor class, 
from processor import Processor
processor = Processor()

In [83]:
# Parse the Fortran source file (e.g., area.f90).
# This reads the file and returns an abstract syntax tree (AST) representation.
tree = processor.parse_fortran_file("./examples/area.f90")
print(tree)

INFO:root:Successfully parsed file: ./examples/area.f90


MODULE main_module
  USE math_utils
  IMPLICIT NONE
  REAL :: area

  CONTAINS

    SUBROUTINE compute_area(radius)
    REAL, INTENT(IN) :: radius
    REAL :: r_squared
    r_squared = radius * radius
    area = pi * r_squared
    CALL increment_counter(1)
  END SUBROUTINE compute_area

END MODULE main_module


In [84]:
# A simple Fortran program/module/proceduer as a string
code = """
subroutine increment_counter(value)
    integer, intent(in) :: value
    global_counter = global_counter + value
end subroutine increment_counter
"""
subroutine_tree = processor.parse_fortran_string(code)
print(subroutine_tree)

INFO:root:Successfully parsed string!



SUBROUTINE increment_counter(value)
  INTEGER, INTENT(IN) :: value
  global_counter = global_counter + value
END SUBROUTINE increment_counter


In [85]:
# A piece of Fortran code as string
code = """
minval_value = HUGE(0.0)
minval_index = 1
DO i = 1, N
  IF (A(i) .LT. minval_value) THEN
    minval_value = A(i)
    minval_index = i
  END IF
END DO
i = minval_index
"""
parsed = processor.parse_fortran_statement(code)
print(parsed)

INFO:root:Successfully parsed statement: 
minval_value = HUGE(0.0)
minval_index = 1
DO i = 1, N
  IF (A(i) .LT. minval_value) THEN
    minval_value = A(i)
    minval_index = i
  END IF
END DO
i = minval_index



minval_value = HUGE(0.0)
minval_index = 1
DO i = 1, N
  IF (A(i) .LT. minval_value) THEN
    minval_value = A(i)
    minval_index = i
  END IF
END DO
i = minval_index


In [86]:
# A Fortran comment/ compiler directives
comment = "!$ACC END PARALLEL"
comment_node = processor.parse_fortran_comment(comment)
print(comment_node)

!$ACC END PARALLEL


In [87]:
# Benchmark directory is an attribute (set it accordingly)
processor.benchmark_dir = "/path/to/benchmark"
# Generate and parse a dummy subroutine named 'my_subroutine'
dummy_subroutine_node = processor.initiate_empty_routine("my_subroutine")
print(dummy_subroutine_node)

SUBROUTINE read_dummy
  OPEN(UNIT = 1363, FILE = '/path/to/benchmark/my_subroutine/dummy.bin', FORM = 'unformatted', STATUS = 'old')
  WRITE(*, *) '--- inside the read dummy routine for my_subroutine ---'
END SUBROUTINE read_dummy


In [88]:
# Sample combined declaration statement
stat = F23.Type_Declaration_Stmt("REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup, Fdn, Fab")
items = processor.separate_entity_declarations(stat)
for decl in items:
    print(decl)
    print("")  

REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup

REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fdn

REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fab



In [89]:
# Define which variables are modified inside the proceduer
var_modif = ["Fup", "b", "c"]
# Modify the first declaration (Fup)
modified_entity_declaration = processor.add_entity_to_declaration(items[0], var_modif)
print(modified_entity_declaration)

REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup, Fup_cpu


In [90]:
# A compound allocate statement with multiple variables and a stat= option
allocate_stmts = F23.Allocate_Stmt("ALLOCATE(a(n), b(m), STAT = ier)")
# Separate the allocate statement into individual allocations
allocated_stmts = processor.separate_entity_allocation(allocate_stmts)
for item in allocated_stmts:
    print(item)

INFO:root:Successfully generated allocation statements


ALLOCATE(a(n), STAT = ier)
ALLOCATE(b(m), STAT = ier)


In [91]:
# Define which variables are modified inside the proceduer
var_modif = ['a','b','c']
add_allocated_stmts = processor.add_entity_to_allocation(item,var_modif)
for item in add_allocated_stmts:
    print(item)

INFO:root:Successfully parsed statement: if(.not. allocated(b))then
ALLOCATE(b(m), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(b_cpu))then
ALLOCATE(b_cpu(m), STAT = ier)
end if
INFO:root:Successfully generated allocation statements


IF (.NOT. ALLOCATED(b)) THEN
  ALLOCATE(b(m), STAT = ier)
END IF
IF (.NOT. ALLOCATED(b_cpu)) THEN
  ALLOCATE(b_cpu(m), STAT = ier)
END IF


In [92]:
# Explicit declaration with known shape
explicit_dec = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(kjpindex) :: evapot")
# Implicit declaration with assumed shape (:)
implicit_dec = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(:) :: ava")
# Map the shape from explicit to implicit
mapped = processor.map_declaration(implicit_dec, explicit_dec=explicit_dec)
print(mapped)

INFO:root:Mapped declaration: REAL(KIND = r_std), DIMENSION(kjpindex) :: ava


REAL(KIND = r_std), DIMENSION(kjpindex) :: ava


In [93]:
# Use the same implicit declaration
implicit_dec = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(:) :: ava")
# Directly specify dimensions as a string
dimensions = "kjpindex, nlev"
# Call map_declaration with explicit dimension override
mapped = processor.map_declaration(implicit_dec, dimensions=dimensions)
print(mapped)

INFO:root:Mapped declaration: REAL(KIND = r_std), DIMENSION(kjpindex, nlev) :: ava


REAL(KIND = r_std), DIMENSION(kjpindex, nlev) :: ava


In [94]:
# Original declaration and allocation as typically found in Fortran
declaration_stmt = F23.Type_Declaration_Stmt("REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:) :: a")
allocate_stmt = F23.Allocate_Stmt("ALLOCATE(a(n), STAT = ier)")
# Combine the two into a single fixed-size declaration
variable_declarations = [allocate_stmt, declaration_stmt]
combined = processor.combine_allocate_declaration(variable_declarations)
print(combined)

INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(n) :: a


REAL(KIND = r_std), DIMENSION(n) :: a


In [95]:
# Original statement with INTENT
declaration_stmt = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(n), INTENT(OUT) :: a")
# Remove INTENT attribute
cleaned_stmts = processor.remove_intent_and_save([declaration_stmt])
# Print results
for stmt in cleaned_stmts:
    print(stmt)

INFO:root:Successfully removed INTENT and SAVE attributes from statements


REAL(KIND = r_std), DIMENSION(n) :: a


In [96]:
# Example 1: Compare two real-valued arrays named `a` and `a_cpu`
processor.check_point('a', 'a_cpu', ['REAL', 'DIMENSION'])

# Example 2: Compare two logical scalar variables named `a` and `a_cpu`
processor.check_point('a', 'a_cpu', ['LOGICAL'])

# Example 3 (optional extension): Compare two logical arrays
processor.check_point('a', 'a_cpu', ['LOGICAL', 'DIMENSION'])

# Example 4 (default case): Compare two scalar real variables
processor.check_point('a', 'a_cpu', ['REAL'])

INFO:root:Successfully parsed statement: 
                IF (ALL(a .EQ. a_cpu)) THEN
                    write(*,*) 'Test passed: All elements in a_gpu are equal to a_cpu.'
                ELSE
                    write(*,*) ''
                    write(*,*) 'Test failed: All elements in a_gpu do not match a_cpu.'
                    write(*,'(A, E25.16)') 'Maximum absolute error:',  maxval(abs(a - a_cpu))
                    write(*,'(A, 2E25.16)') 'Min and Max of a_gpu:', minval(a), maxval(a)
                    write(*,'(A, 2E25.16)') 'Min and Max of a_cpu:', minval(a_cpu), maxval(a_cpu)
                    write(*,*) ''
                ENDIF
                
INFO:root:Successfully parsed statement: 
                IF (a .EQV. a_cpu) THEN
                    write(*,*) 'LOGICAL EQV test passed: a_gpu is equal to a_cpu.'
                ELSE
                    write(*,*) ''
                    write(*,*) 'LOGICAL EQV test failed: a_gpu does not match a_cpu.'
                    wr

Execution_Part(If_Construct(If_Then_Stmt(Level_4_Expr(Name('a'), '.EQ.', Name('a_cpu'))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("'Test passed: a_gpu is equal to a_cpu.'", None),))), Else_Stmt(None), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("''", None),))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("'Test failed: a_gpu does not match a_cpu.'", None),))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Char_Literal_Constant("'(A, E25.16)'", None)))), Output_Item_List(',', (Char_Literal_Constant("'Absolute error:'", None), Intrinsic_Function_Reference(Intrinsic_Name('ABS'), Actual_Arg_Spec_List

In [97]:
# Create a CALL statement from a parsed subroutine tree
# This is useful when you want to generate a call to a subroutine whose AST you have already parsed.
call_stmt = processor.create_call_stmt(subroutine_tree)
print(call_stmt)

CALL increment_counter(value)


## Extractor and Isolator classes testing
Some methods in the `Processor` class depend on other classes that must be initialized before using.
Once set up, you can use and test individual methods of the class interactively in the notebook environment.

In [98]:
# Define the path to the Fortran module we want to analyze.
# This is the subdirectory (relative or absolute) where the source code resides.
rest_of_path = "modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"

# Define the name of the target Fortran module (without the .f90 extension).
target_module = "hydrol"

# Retrieve the base working directory from the environment variable `works`.
# This should be set in your shell or notebook environment beforehand.
work = os.getenv("works")

# Full path to the target module
full_module_path = os.path.join(work, rest_of_path, f"{target_module}.f90")

print("Full module path:", full_module_path)

Full module path: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90


In [99]:
# Create an instance of the Isolator and Extractor classes

from isolator import Isolator
from extractor import Extractor

# The Isolator class prepares the environment to isolate a Fortran procedure
# from a full codebase so that it can be run independently.
openacc = False
isolator = Isolator(rest_of_path, target_module, work, openacc)

# The Extractor class takes the parsed Fortran module tree from the isolator
# and extracts all necessary metadata such as subroutine calls, variable declarations,
# dummy arguments, and other dependencies.
extractor = Extractor(isolator.module_dir_sp, isolator.module_tree_cp)

INFO:root:Successfully parsed file: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.f90
INFO:root:Successfully parsed string!


In [100]:
# Extract subroutines and gather their metadata
extractor.find_subroutines()

# List all identified subroutine names in the module
extractor.subroutine_keys_all

{'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_muff',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_coef_setup',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr',
 'hydrol_soil_tridiag',
 'hydrol_split_soil',
 'hydrol_tmc_update',
 'hydrol_vegupd'}

In [101]:
# loops info
extractor.extract_loop_indices() 
extractor.loop_dict

defaultdict(set,
            {'nslm': {'isl', 'jsl'},
             'kjpindex': {'ipts', 'ji'},
             'nvm': {'ivm', 'jv'},
             'nstm': {'ist', 'jst'},
             'itopmax': {'jsl'},
             'nslm - 1': {'jsl'},
             'imax - 1': {'ii'},
             'imin': {'ii'},
             '4': {'jsl'},
             'nslm - 2': {'jsl'},
             '2': {'jsl'},
             '1': {'jrp', 'jsl'},
             'nbp_glo': {'ji'},
             'nsnow': {'jg'},
             'nrp': {'jrp'},
             'nrp - 1': {'jrp'}})

In [102]:
# Call statement done within a subroutine
extractor.call_within_sub 
for parent in extractor.call_within_sub.keys():
    print('\n')
    print(f'Parent: {parent}, children: {extractor.call_within_sub[parent]}')



Parent: hydrol_main, children: {'hydrol_hydraulic_arch_tuzet_calc', 'explicitsnow_main', 'hydrol_vegupd', 'hydrol_canop', 'hydrol_nudge_mc_diag', 'histwrite_p', 'hydrol_soil', 'hydrol_nudge_snow', 'hydrol_flood', 'hydrol_alma'}


Parent: hydrol_vegupd, children: {'hydrol_tmc_update'}


Parent: hydrol_soil, children: {'hydrol_soil_setup', 'hydrol_split_soil', 'hydrol_soil_froz', 'hydrol_diag_soil_flux', 'hydrol_root_profile', 'hydrol_soil_smooth_over_mcs2', 'hydrol_nudge_mc', 'hydrol_soil_coef', 'hydrol_soil_smooth_under_mcr', 'hydrol_soil_tridiag', 'hydrol_diag_soil', 'hydrol_soil_infilt'}


Parent: hydrol_nudge_snow, children: {'xios_orchidee_recv_field', 'scatter', 'flininfo', 'flinget'}


Parent: hydrol_hydraulic_arch_tuzet_calc, children: {'hydrol_hydraulic_arch_tuzet_resist', 'hydrol_hydraulic_arch_tuzet_muff'}


Parent: hydrol_hydraulic_arch_tuzet_muff, children: {'hydrol_muff_radial_coef_setup'}


Parent: hydrol_muff_radial_coef_setup, children: {'hydrol_muff_radial_resolution

Now, we examine the subroutines (children) that are called within other subroutines (parents). During isolation, the process begins with the most deeply nested child subroutines and proceeds outward, gradually isolating higher-level parent subroutines that depend on them.

In [103]:
# we will begin with seeing a subroutine of hydrol_soil
subroutine_key = 'hydrol_diag_soil'

In [104]:
# Retrieve the full AST node (subroutine as a Fparser Subroutine_Subprogram object)
subroutine_tree = extractor.subroutines[subroutine_key]

# Re-parse the subroutine into a new AST for further analysis or transformation
parsed_subroutine_tree = processor.parse_fortran_string(str(subroutine_tree))

# Analyze the usage of dummy arguments within a Fortran subroutine, None means that the variable is in the dummy, but not used. 
extractor.extract_intent(subroutine_key, subroutine_tree)
print(extractor.general_usage_dict[subroutine_key])
# Validates and corrects INTENT specifications
extractor.clean_subroutine(subroutine_key, subroutine_tree)

INFO:root:Successfully parsed string!


{'ks': 'IN', 'nvan': None, 'avan': None, 'mcr': None, 'mcs': 'IN', 'mcfc': 'IN', 'mcw': 'IN', 'kjpindex': 'IN', 'veget_max': 'IN', 'soiltile': 'IN', 'njsc': None, 'runoff': 'OUT', 'drainage': 'OUT', 'evapot': None, 'vevapnu': 'INOUT', 'returnflow': 'IN', 'reinfiltration': 'IN', 'irrigation': 'IN', 'shumdiag': 'OUT', 'shumdiag_perma': 'OUT', 'k_litt': 'OUT', 'litterhumdiag': 'OUT', 'humrel': 'OUT', 'vegstress': 'OUT', 'drysoil_frac': 'OUT', 'tot_melt': 'IN', 'us': 'INOUT', 'precip_rain': 'IN', 'totfrac_nobio': 'IN', 'frac_snow_nobio': 'IN'}


In [105]:
# extract and categorize variables
extractor.find_variables(subroutine_tree,subroutine_key)
extractor.extract_names(subroutine_key)

# Print categorized variables for inspection
print("🧩 Declared Variables:")
print(extractor.var_declared[subroutine_key])

print("\n📥 Dummy Arguments (with intent):")
for stmt in extractor.var_dummy[subroutine_key]:
    print(stmt.tostr())

print("\n📦 Local Variables (non-dummy, declared):")
for stmt in extractor.var_local[subroutine_key]:
    print(stmt.tostr())

print("\n🌍 Global Variables (used but not declared):")
print(extractor.var_global[subroutine_key])

print("\n🛠️ Modified Variables (on LHS of assignment):")
print(extractor.var_modif[subroutine_key])

print("\n📐 Implicitly Shaped Variables (transformed to explicit):")
for name, decl in extractor.imp_shape[subroutine_key].items():
    print(f"{name} -> {decl.tostr()}")


🧩 Declared Variables:
{'us', 'mcr', 'mcfc', 'humrel', 'soiltile', 'mask_vegtot', 'kjpindex', 'returnflow', 'avan', 'njsc', 'nvan', 'frac_snow_nobio', 'ks', 'i', 'jst', 'irrigation', 'totfrac_nobio', 'shumdiag', 'ji', 'tmc_litter_ratio', 'litterhumdiag', 'drainage', 'k_tmp', 'shumdiag_perma', 'precip_rain', 'reinfiltration', 'mcw', 'k_litt', 'evapot', 'veget_max', 'mcs', 'jsl', 'runoff', 'tot_melt', 'drysoil_frac', 'vevapnu', 'jv', 'vegstress'}

📥 Dummy Arguments (with intent):
REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: avan
REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(OUT) :: drainage
REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(OUT) :: drysoil_frac
REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: evapot
REAL(KIND = r_std), DIMENSION(kjpindex, nnobio), INTENT(IN) :: frac_snow_nobio
REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(OUT) :: humrel
REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: irrigation
REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(O

In [106]:
# Recursively searches for the declarations of global variables and external procedures 
extractor.find_global_variables(isolator.module_dir_sp, isolator.module_tree_sp, extractor.var_global[subroutine_key], subroutine_key)

Searching for variable: soilmoist_s ... ⏳
<soilmoist_s> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:, :, :) :: soilmoist_s
<soilmoist_s> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(soilmoist_s(kjpindex, nslm, nstm), STAT = ier)
The containing directory is: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/
✅ Variable found!

Searching for variable: vegtot ... ⏳
<vegtot> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:) :: vegtot
<vegtot> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(vegtot(kjpindex), STAT = ier)
The containing directory is: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/
✅ Variable found!

Searching for variable: tmc_litt_wet_mea ... ⏳
<tmc_litt_wet_mea> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:) :: tmc_litt_wet_mea
<tmc_li

INFO:root:Successfully parsed file: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_parallel/xios_orchidee.f90


Add! Module xios is added into the queue.
Add! Module defprec is added into the queue.
Add! Module pft_parameters_var is added into the queue.
Add! Module constantes_var is added into the queue.
Add! Module constantes_soil_var is added into the queue.
Add! Module vertical_soil_var is added into the queue.
Add! Module IOIPSL is added into the queue.
Add! Module mod_orchidee_para_var is added into the queue.
Add! Module mod_orchidee_transfert_para is added into the queue.
Add! Module ioipsl_para is added into the queue.
Checking the child module .... constantes


INFO:root:Successfully parsed file: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters/constantes.f90
INFO:root:Successfully parsed file: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_global/time.f90


Checking the child module .... time
Add! Module function_library is added into the queue.
Checking the child module .... constantes_soil


...

Checking the child module .... constantes_var


INFO:root:Successfully parsed file: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters/constantes_var.f90


Checking the child module .... constantes_soil_var


INFO:root:Successfully parsed file: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters/constantes_soil_var.f90


<imin> is found in <<constantes_soil_var>> of the module <<< constantes_soil_var >>>
INTEGER(KIND = i_std), PARAMETER :: imin = 1
The containing directory is: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters
✅ Variable found!

Searching for variable: dr_ns ... ⏳
<dr_ns> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:, :) :: dr_ns
<dr_ns> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(dr_ns(kjpindex, nstm), STAT = ier)
The containing directory is: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/
✅ Variable found!

Searching for variable: soil_wet_litter ... ⏳
<soil_wet_litter> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:, :) :: soil_wet_litter
<soil_wet_litter> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(soil_wet_litter(kjpindex, nstm), STAT = ier)
The containing directory is: /scratchu/kardaneh/mo

In [107]:
# Process dummy argument declarations to extract shape and scalar information
extractor.process_declaration_variables(extractor.var_dummy[subroutine_key], subroutine_key)

# Process global declarations retrieved from other modules (already stored in dec_global)
for key in extractor.dec_global[subroutine_key].keys():
    extractor.process_declaration_variables(extractor.dec_global[subroutine_key][key], subroutine_key)

# Identify shape-related variables that still need to be searched in other modules
# We subtract scalar and already-known global variables from all shape variables
shape_to_search = extractor.shapes_variables[subroutine_key] - extractor.scalar_variables[subroutine_key] - extractor.var_global[subroutine_key]

# If there are still unresolved shape variables, search them in other modules
if shape_to_search:
    extractor.find_global_variables(isolator.module_dir, isolator.module_tree, shape_to_search, subroutine_key)
    extractor.var_global[subroutine_key].update(shape_to_search)


print(" Scalar Variables:")
print(extractor.scalar_variables[subroutine_key])

print("\n Shape Variables:")
print(extractor.shapes_variables[subroutine_key])

print("\n Variables to search in other modules (shape_to_search):")
print(shape_to_search)

print("\n Final Global Variables (after update):")
print(extractor.var_global[subroutine_key])

 Scalar Variables:
{'ok_freeze_cwrr', 'min_sechiba', 'zero', 'imin', 'huit', 'un', 'imax', 'nnobio', 'iice', 'trois'}

 Shape Variables:
{'nnobio', 'imax', 'imin'}

 Variables to search in other modules (shape_to_search):
set()

 Final Global Variables (after update):
{'soilmoist_s', 'vegtot', 'tmc_litt_wet_mea', 'imin', 'dr_ns', 'soil_wet_litter', 'tmc', 'profil_froz_hydro_ns', 'humtot', 'mc', 'vegtot_old', 'dz', 'min_sechiba', 'tmc_litter_res', 'subsinksoil', 'ru_ns', 'k_lin', 'un', 'trois', 'ok_freeze_cwrr', 'tmc_litter_adry', 'vegstressv', 'mask_soiltile', 'humrelv', 'tmc_litt_dry_mea', 'soil_wet_ns', 'imax', 'soilmoist', 'tmc_litter_sat', 'tmc_litt_mea', 'zero', 'ae_ns', 'soilmoist_liquid', 'tmc_litter', 'huit', 'frac_bare_ns', 'nnobio', 'iice', 'profil_froz_hydro', 'dh', 'tmc_litter_awet', 'mcl'}


In [108]:
# Extract array shape information for the current subroutine
# This includes arrays from:
#   - Global declarations (cls.dec_global)
#   - Dummy arguments (cls.var_dummy)
#   - Local variables (cls.var_local, handled internally)

extractor.extract_array_info(
    extractor.dec_global[subroutine_key],  # External/global declarations
    extractor.var_dummy[subroutine_key],   # Dummy argument declarations
    subroutine_key                   # Current subroutine identifier
)

# 
print(f"\n Array shape info for subroutine '{subroutine_key}':")
for var_name, dims in extractor.all_array_info[subroutine_key].items():
    print(f" - {var_name}:")
    for i, dim in enumerate(dims):
        print(f"    Dim {i+1}: Start = {dim['dim_str']}, End = {dim['dim_end']}")


print(f"\n Modified variable info for subroutine '{subroutine_key}':")
for var, info in extractor.var_modif_info[subroutine_key].items():
    print(f" - {var}: {info}")

INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: soilmoist_s
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_wet_mea
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: dr_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: soil_wet_litter
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tmc
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: profil_froz_hydro_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: humtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: mc
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot_old
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(nslm) :: dz
INFO:root:Combined statement: REAL(KIND 


 Array shape info for subroutine 'hydrol_diag_soil':
 - soilmoist_s:
    Dim 1: Start = 1, End = kjpindex
    Dim 2: Start = 1, End = nslm
    Dim 3: Start = 1, End = nstm
 - vegtot:
    Dim 1: Start = 1, End = kjpindex
 - tmc_litt_wet_mea:
    Dim 1: Start = 1, End = kjpindex
 - dr_ns:
    Dim 1: Start = 1, End = kjpindex
    Dim 2: Start = 1, End = nstm
 - soil_wet_litter:
    Dim 1: Start = 1, End = kjpindex
    Dim 2: Start = 1, End = nstm
 - tmc:
    Dim 1: Start = 1, End = kjpindex
    Dim 2: Start = 1, End = nstm
 - profil_froz_hydro_ns:
    Dim 1: Start = 1, End = kjpindex
    Dim 2: Start = 1, End = nslm
    Dim 3: Start = 1, End = nstm
 - humtot:
    Dim 1: Start = 1, End = kjpindex
 - mc:
    Dim 1: Start = 1, End = kjpindex
    Dim 2: Start = 1, End = nslm
    Dim 3: Start = 1, End = nstm
 - vegtot_old:
    Dim 1: Start = 1, End = kjpindex
 - dz:
    Dim 1: Start = 1, End = nslm
 - tmc_litter_res:
    Dim 1: Start = 1, End = kjpindex
    Dim 2: Start = 1, End = nstm
 - sub

In [109]:
# Extract the vectorized loop from the subroutine (if it uses 'kjpindex' as loop upper bound)
extractor.extract_loop_vect(subroutine_key, subroutine_tree)

# Print the extracted vector loop structure for inspection
if extractor.loop_vect[subroutine_key]:
    print(f"\033[32mExtracted vector loop for '{subroutine_key}':\033[0m\n{extractor.loop_vect[subroutine_key]}")
else:
    print(f"\033[33mNo vector loop (ending with 'kjpindex') found in subroutine '{subroutine_key}'.\033[0m")

Extracted vector loop for 'hydrol_diag_soil':
DO ji = 1, kjpindex


In [110]:
# Process global declarations and modifications for integration into:
#         - The declaration module
#         - The read/write routines
#         - OpenACC (COPYIN / CREATE) directives
#         - Allocation statements
#
# This step finalizes how each variable is introduced 

processor.add_declarations(
    extractor.dec_global[subroutine_key],              # All globally extracted variable declarations
    extractor.var_modif_info[subroutine_key],           # Mapping of modified variables and their metadata
    openacc=openacc
)



print("\n\033[1;32m✅ Declarations added to module:\033[0m")
for stmt in processor.add_to_module:
    print(stmt.tostr())

print("\n\033[1;36m📌 Allocation statements to add to routine:\033[0m")
for stmt in processor.add_to_routin:
    print(stmt.tostr())

print("\n\033[1;35m📥 Read Statements in declaration routine:\033[0m")
for stmt in processor.reads_in_decleration_routine:
    print(stmt.tostr())

print("\n\033[1;35m📥 Read Statements in separate read routine:\033[0m")
for stmt in processor.reads_in_read_routine:
    print(stmt.tostr())

print("\n\033[1;34m📤 Write Statements for writing to file:\033[0m")
for stmt in processor.write_stmt:
    print(stmt)


if processor.acc_declare_copyin:
    print("\n\033[1;33m🚀 OpenACC CREATE directive:\033[0m")
    print(processor.acc_declare_create_cmd.tostr())

if processor.acc_declare_create:
    print("\n\033[1;33m🚀 OpenACC COPYIN directive:\033[0m")
    print(processor.acc_declare_copyin_cmd.tostr())


INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns
INFO:root:Successfully parsed statement: if(.not. allocated(ae_ns))then
ALLOCATE(ae_ns(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(ae_ns_cpu))then
ALLOCATE(ae_ns_cpu(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully generated allocation statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(nslm) :: dh
INFO:root:Successfully parsed statement: if(.not. allocated(dh))then
ALLOCATE(dh(nslm), STAT = ier)
end if
INFO:root:Successfully generated allocation statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: dr_ns
INFO:root:Successfully parsed statement: if(.not. allocated(dr_ns))then
ALLOCATE(dr_ns(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(dr_ns_cpu))then
ALLOCATE(dr_ns_cpu(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully generated 


✅ Declarations added to module:
REAL(KIND = r_std), PARAMETER :: zero = 0._r_std
REAL(KIND = r_std), PARAMETER :: un = 1._r_std
REAL(KIND = r_std), PARAMETER :: trois = 3._r_std
INTEGER(KIND = i_std), PARAMETER :: nnobio = 1
REAL(KIND = r_std), PARAMETER :: min_sechiba = 1.E-8_r_std
INTEGER(KIND = i_std), PARAMETER :: imin = 1
INTEGER(KIND = i_std), PARAMETER :: iice = 1
REAL(KIND = r_std), PARAMETER :: huit = 8._r_std
INTEGER :: imax
LOGICAL :: ok_freeze_cwrr
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:, :) :: ae_ns, ae_ns_cpu
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:) :: dh
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:, :) :: dr_ns, dr_ns_cpu
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:) :: dz
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:, :) :: frac_bare_ns
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:, :, :) :: humrelv, humrelv_cpu
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:) :: humtot, humtot_cpu
REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:, :, :) :: k_lin
INTEGER(KIND = i_std),

In [111]:
# Update the global module by injecting declarations, use statements,
# and I/O operations related to global variables for the specified subroutine.
subroutine_dir = os.path.join(isolator.target_module_dir, subroutine_key)
file_path = os.path.join(subroutine_dir, isolator.module_global_file)
print(f"Updating global module for subroutine '{subroutine_key}' at '{file_path}'...")
processor.update_global_module(
    extractor.dec_global[subroutine_key],
    file_path,
    subroutine_key,
    isolator.module_tree_cp
)
print(f"Global module update completed for subroutine '{subroutine_key}'.")
print(processor.out_module)

INFO:root:Successfully parsed module code
INFO:root:Successfully parsed statement: CALL hydrol_diag_soil(ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio)
open(unit=1363, file='/path/to/benchmark/hydrol_diag_soil/global.bin', form='unformatted', status='replace')
WRITE(1363) imax
WRITE(1363) ok_freeze_cwrr
WRITE(1363) ae_ns
WRITE(1363) dh
WRITE(1363) dr_ns
WRITE(1363) dz
WRITE(1363) frac_bare_ns
WRITE(1363) humrelv
WRITE(1363) humtot
WRITE(1363) k_lin
WRITE(1363) mask_soiltile
WRITE(1363) mc
WRITE(1363) mcl
WRITE(1363) profil_froz_hydro
WRITE(1363) profil_froz_hydro_ns
WRITE(1363) ru_ns
WRITE(1363) soil_wet_litter
WRITE(1363) soil_wet_ns
WRITE(1363) soilmoist
WRITE(1363) soilmoist_liquid
WRITE(1363) soilmoist_s
WRITE(1363) subsinksoil
WRITE(1363) tmc

Updating global module for subroutine 'hydrol_diag_soil' at '/home/kardaneh/Fgpt/hydrol/hydrol_diag_soil/module_global.f90'...
Global module update completed for subroutine 'hydrol_diag_soil'.

MODULE module_global
  IMPLICIT NONE
  INTEGER, PARAMETER :: i_std = 4
  INTEGER, PARAMETER :: r_std = 8
  INTEGER(KIND = i_std), PARAMETER :: nsnow = 3
  INTEGER(KIND = i_std), PARAMETER :: nslm = 11
  INTEGER(KIND = i_std), PARAMETER :: nvm = 15
  INTEGER(KIND = i_std), PARAMETER :: nstm = 3
  INTEGER(KIND = i_std), PARAMETER :: kjpindex = 4717
  INTEGER :: ier
  INTEGER(KIND = i_std) :: ic0, ic
  REAL(KIND = r_std) :: icr, start_time, stop_time
  REAL(KIND = r_std), PARAMETER :: zero = 0._r_std
  REAL(KIND = r_std), PARAMETER :: un = 1._r_std
  REAL(KIND = r_std), PARAMETER :: trois = 3._r_std
  INTEGER(KIND = i_std), PARAMETER :: nnobio = 1
  REAL(KIND = r_std), PARAMETER :: min_sechiba = 1.E-8_r_std
  INTEGER(KIND = i_std), PARAMETER :: imin = 1
  INTEGER(KIND = i_std), PARAMETER :: iice = 

In [112]:
# Prepare main program update
#
# This section constructs the original call statement to the subroutine
# and builds the list of subroutine trees and corresponding call 
# statements that will be inserted into the main program. If `openacc` 
# is enabled, the modified subroutine tree and its associated call 
# statement (vectorized version) are also added.
#
# These components are then passed to `update_main_program`, which 
# rewrites the main program to include performance measurement, 
# OpenACC directives (if enabled), error checks, and binary I/O for 
# benchmark data generation.


file_path = os.path.join(subroutine_dir, isolator.main_program_file)

# Create original call and subroutine tree
sub_trees = [subroutine_tree]
arg_list = ', '.join([name for name in extractor.dummy_arg_list[subroutine_key]])
call_stmt_org = F23.Call_Stmt(f"CALL {subroutine_key}({arg_list})")
call_stmts = [call_stmt_org]

# If OpenACC is enabled, append the modified tree and call statement
if openacc:
    sub_trees.append(modified_subroutine_tree)
    call_stmts.append(call_stmt_vec)

# Update the main program with all gathered components
processor.update_main_program(
    custom_dec_inout=extractor.var_dummy[subroutine_key],
    custom_subroutine_trees=sub_trees,
    call_stmts=call_stmts,
    var_modif=extractor.var_modif_info[subroutine_key],
    file_path=file_path,
    subroutine_name=subroutine_key,
    dummy_args=extractor.dummy_arg_list[subroutine_key],
    module_tree=isolator.module_tree_cp,
    childs_subroutine_tree=None,
    openacc=openacc,
    dummy_add_decl=None,
    error_flag=None,
    acc_data_copyin=None
)

# Print the updated main program Fortran code
print(processor.out_main)


INFO:root:Successfully parsed main program code
INFO:root:Successfully parsed statement: 
                    read(1363, iostat = ier)avan
                    if (ier /= 0) then
                    write(*,*) 'Error reading from file for avan. ',' IOSTAT : ', ier
                    endif
                    
INFO:root:Successfully parsed statement: 
                    read(1363, iostat = ier)evapot
                    if (ier /= 0) then
                    write(*,*) 'Error reading from file for evapot. ',' IOSTAT : ', ier
                    endif
                    
INFO:root:Successfully parsed statement: 
                    read(1363, iostat = ier)frac_snow_nobio
                    if (ier /= 0) then
                    write(*,*) 'Error reading from file for frac_snow_nobio. ',' IOSTAT : ', ier
                    endif
                    
INFO:root:Successfully parsed statement: 
                    read(1363, iostat = ier)irrigation
                    if (ier /= 0) then
 

need to build an initialization for in/inout dummy args: 

PROGRAM main
  USE module_global
  IMPLICIT NONE
  REAL(KIND = r_std), DIMENSION(kjpindex) :: avan
  REAL(KIND = r_std), DIMENSION(kjpindex) :: drainage, drainage_cpu
  REAL(KIND = r_std), DIMENSION(kjpindex) :: drysoil_frac, drysoil_frac_cpu
  REAL(KIND = r_std), DIMENSION(kjpindex) :: evapot
  REAL(KIND = r_std), DIMENSION(kjpindex, nnobio) :: frac_snow_nobio
  REAL(KIND = r_std), DIMENSION(kjpindex, nvm) :: humrel, humrel_cpu
  REAL(KIND = r_std), DIMENSION(kjpindex) :: irrigation
  REAL(KIND = r_std), DIMENSION(kjpindex) :: k_litt, k_litt_cpu
  REAL(KIND = r_std), DIMENSION(kjpindex) :: ks
  REAL(KIND = r_std), DIMENSION(kjpindex) :: litterhumdiag, litterhumdiag_cpu
  REAL(KIND = r_std), DIMENSION(kjpindex) :: mcfc
  REAL(KIND = r_std), DIMENSION(kjpindex) :: mcr
  REAL(KIND = r_std), DIMENSION(kjpindex) :: mcs
  REAL(KIND = r_std), DIMENSION(kjpindex) :: mcw
  INTEGER(KIND = i_std), DIMENSION(kjpindex) :: njsc
  REAL(KIND 

The executive.py is the file that should be run after compiling and simulating the ORCHIDEE binary. Which runs the compile_and_run of the 
processor class which will use the benchmark dataset to run and test out the modified fortran files. 

In [113]:
from f2np import F2NP

f2np_ = F2NP() # THe f2np class is in charge of translating the fortran code to python, it primarily goes through the recursive method which 
# gets a block of fortran code to translte.

# It effectively does by using verifying each child of the block and breaking them into the different blocks and then sending them to the 
# the correct method in question. As such a subroutine_stmt which contains a subroutines will be sent to the handle_subroutine_stmt method,
# a Do or If statement are sent respectively to the handle_do_stmt and handle_assignements 
# This ways it translates each line onto a python code.

In [114]:
f2np_.recursive(subroutine_tree) # Hence we can see that each line is translated from fortran code to python code based on the
# their instances(declaration and statements)

SUBROUTINE hydrol_diag_soil(ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio)
def hydrol_diag_soil(ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio):

INTEGER(KIND = i_std), INTENT(IN) :: kjpindex
None

REAL(KIND = r_std), DIMENSION(kjpindex, nvm), INTENT(IN) :: veget_max
None

INTEGER(KIND = i_std), DIMENSION(kjpindex), INTENT(IN) :: njsc
None

REAL(KIND = r_std), DIMENSION(kjpindex, nstm), INTENT(IN) :: soiltile
None

REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) :: evapot
None

REAL(KIND = r_std), DIMENSIO